In [1]:
#!pip install librosa tensorflow scikit-learn

import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
!git clone https://github.com/Jakobovski/free-spoken-digit-dataset.git

Cloning into 'free-spoken-digit-dataset'...
Updating files:  21% (655/3014)
Updating files:  22% (664/3014)
Updating files:  23% (694/3014)
Updating files:  24% (724/3014)
Updating files:  25% (754/3014)
Updating files:  26% (784/3014)
Updating files:  27% (814/3014)
Updating files:  28% (844/3014)
Updating files:  29% (875/3014)
Updating files:  30% (905/3014)
Updating files:  31% (935/3014)
Updating files:  32% (965/3014)
Updating files:  33% (995/3014)
Updating files:  34% (1025/3014)
Updating files:  35% (1055/3014)
Updating files:  36% (1086/3014)
Updating files:  37% (1116/3014)
Updating files:  38% (1146/3014)
Updating files:  39% (1176/3014)
Updating files:  40% (1206/3014)
Updating files:  41% (1236/3014)
Updating files:  42% (1266/3014)
Updating files:  43% (1297/3014)
Updating files:  44% (1327/3014)
Updating files:  45% (1357/3014)
Updating files:  45% (1369/3014)
Updating files:  46% (1387/3014)
Updating files:  47% (1417/3014)
Updating files:  48% (1447/3014)
Updating fil

In [3]:
DATA_PATH = "free-spoken-digit-dataset/recordings"

SAMPLE_RATE = 8000
MFCC_NUM = 13
MAX_LEN = 32

def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=MFCC_NUM)
    
    if mfcc.shape[1] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0,0),(0,MAX_LEN - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :MAX_LEN]
    
    return mfcc

In [4]:
X = []
y = []
file_paths = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".wav"):
        label = int(file.split("_")[0])
        path = os.path.join(DATA_PATH, file)
        
        X.append(extract_features(path))
        y.append(label)
        file_paths.append(path)

X = np.array(X)[..., np.newaxis]
y = np.array(y)

X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X, y, file_paths, test_size=0.2, random_state=42
)

C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1933
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1399
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1876
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1983
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2033
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(13, 32, 1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=15, batch_size=16, validation_split=0.1)

C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.5079 - loss: 2.0937 - val_accuracy: 0.7625 - val_loss: 0.7380
Epoch 2/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8523 - loss: 0.4699 - val_accuracy: 0.9083 - val_loss: 0.3108
Epoch 3/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9310 - loss: 0.2309 - val_accuracy: 0.9208 - val_loss: 0.2103
Epoch 4/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9491 - loss: 0.1606 - val_accuracy: 0.9417 - val_loss: 0.2113
Epoch 5/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9704 - loss: 0.1032 - val_accuracy: 0.9458 - val_loss: 0.1355
Epoch 6/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9699 - loss: 0.0928 - val_accuracy: 0.9708 - val_loss: 0.1026
Epoch 7/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9852 - loss: 0.0534 - val_accuracy: 0.9708 - val_loss: 0.0897
Epoch 8/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9833 - loss: 0.0542 - val_accuracy: 0

In [6]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

model.save("digit_asr_model.h5")

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9617 - loss: 0.1286


Test Accuracy: 0.9616666436195374


In [7]:
model = tf.keras.models.load_model("digit_asr_model.h5")

In [ ]:
import os
import random
import numpy as np
from IPython.display import Audio, display

# -------- COMMON FUNCTION --------
def predict_digit(file_path):
    mfcc = extract_features(file_path)
    mfcc = mfcc[np.newaxis, ..., np.newaxis]
    
    pred = model.predict(mfcc, verbose=0)
    return np.argmax(pred)

# =========================================================
#  OPTION 1: RANDOM AUDIO FROM DATASET
# =========================================================

print("\n--- RANDOM DATASET TEST ---\n")

files = [f for f in os.listdir(DATA_PATH) if f.endswith(".wav")]
random_file = random.choice(files)
file_path = os.path.join(DATA_PATH, random_file)

print("File:", random_file)

display(Audio(file_path))

predicted = predict_digit(file_path)
actual = int(random_file.split("_")[0])

print("Actual Digit:", actual)
print("Predicted Digit:", predicted)
print("-" * 40)


# =========================================================
# OPTION 2: USE YOUR OWN AUDIO FILE
# COMMENT OPTION 1 AND UNCOMMENT BELOW
# =========================================================

# print("\n--- CUSTOM AUDIO TEST ---\n")

# file_path = input("Enter full path of .wav file: ")

# if not os.path.exists(file_path):
#     print("File not found!")
# else:
#     display(Audio(file_path))

#     predicted = predict_digit(file_path)

#     print("Predicted Digit:", predicted)
#     print("-" * 40)


--- RANDOM DATASET TEST ---

File: 6_lucas_14.wav


Actual Digit: 6
Predicted Digit: 6
----------------------------------------
